# **INIT_LOAD_FLAG == 0 IS INCREMNTAL LOAD
# INIT_LOAD_FLAG == 1 IS INTIAL LOAD**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

### Widget for init load flag (1 is initial, 0 is incremntal)

In [0]:
dbutils.widgets.text("init_load_flag", "")
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

### read the silver table and deduplicate

In [0]:
df = spark.read.table("databricks_cata.silver.customers_silver")

In [0]:
df = df.dropDuplicates(['customer_id'])

In [0]:
df.limit(5).display()

### This is the key addition vs SCD1. SCD1 just overwrites the row on a match.
For SCD2 we need to know **if anything actually changed** before deciding to create a new version.
We hash the columns we want to track history for (email, city, state, domains).

In [0]:
df = df.withColumn("row_hash", md5(concat_ws("||", col("email"), col("city"), col("state"), col("domains"))))

In [0]:
df.limit(5).display()

## Step 5: Read existing gold SCD2 table (or empty frame on init load) — same pattern as SCD1

Difference: we only pull the **currently active** version of each customer (`is_current = 1`),
and we also pull `row_hash` so we can compare it against the incoming hash.

In [0]:
if init_load_flag == 0:
    df_old = spark.sql('''select DimCustomerKey, customer_id, row_hash, create_date, update_date
                          from databricks_cata.gold.DimCustomersscd2
                          where is_current = 1''')
else:
    df_old = spark.sql('''select 0 DimCustomerKey, 0 customer_id, 0 row_hash, 0 create_date, 0 update_date
                            from databricks_cata.silver.customers_silver
                            where 1=0''')

In [0]:
df_old.display()

### renaming old columns before joining

In [0]:
df_old = df_old.withColumnRenamed("DimCustomerKey", "old_DimCustomerKey")\
                .withColumnRenamed("customer_id", "old_customer_id")\
                .withColumnRenamed("row_hash", "old_row_hash")\
                .withColumnRenamed("create_date", "old_create_date")\
                .withColumnRenamed("update_date", "old_update_date")

In [0]:
df_old.display()

## Step 7: Join incoming data with the old active records (same join as SCD1)

In [0]:
df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, "left")
df_join.limit(5).display()

## Step 8: NEW for SCD2 — split into 3 buckets instead of 2

SCD1 only had `df_new` (no match) and `df_old` (match → update in place).
SCD2 needs **three** buckets:
1. `df_new` — brand new customers (no match at all)
2. `df_changed` — matched, but `row_hash` is different → needs a new version + expire the old one
3. `df_unchanged` — matched, hash is identical → nothing to do, skip entirely

In [0]:
df_new = df_join.filter(df_join.old_DimCustomerKey.isNull())
df_new.limit(5).display()

In [0]:
df_changed = df_join.filter(df_join.old_DimCustomerKey.isNotNull() & (df_join.row_hash != df_join.old_row_hash))
df_changed.limit(5).display()

In [0]:
df_unchanged = df_join.filter(df_join.old_DimCustomerKey.isNotNull() & (df_join.row_hash == df_join.old_row_hash))
print("unchanged count(no action needed):", df_unchanged.count())

In [0]:
df_expire = df_changed.select(col("old_DimCustomerKey").alias("DimCustomerKey"))\
    .withColumn("merge_key", col("DimCustomerKey").cast("string"))

## Step 10: Build the "new version" rows for changed customers (mirrors your SCD1 df_old rebuild step, but creates a fresh row instead of editing in place)

In [0]:
df_changed_new_version = df_changed.select(df_changed.customer_id, df_changed.email,
                                           df_changed.city, df_changed.state, df_changed.domains,
                                           df_changed.row_hash)

df_changed_new_version = df_changed_new_version\
    .withColumn("effective_start_date", current_timestamp())\
    .withColumn("effective_end_date", lit(None).cast("timestamp"))\
    .withColumn("is_current", lit(1))\
    .withColumn("create_date", current_timestamp())\
    .withColumn("update_date", current_timestamp())


In [0]:
df_changed_new_version.display()

## Step 11: Build the new-customer rows (same idea as your SCD1 df_new rebuild)

In [0]:
df_new = df_new.select(
    df_new.customer_id, df_new.email, df_new.city,
    df_new.state, df_new.domains, df_new.row_hash
)
df_new = df_new\
    .withColumn("effective_start_date", current_timestamp())\
    .withColumn("effective_end_date", lit(None).cast("timestamp"))\
    .withColumn("is_current", lit(1))\
    .withColumn("create_date", current_timestamp())\
    .withColumn("update_date", current_timestamp())

In [0]:
df_new.display()

## Step 12: Union all rows that need a brand-new surrogate key (new customers + new versions of changed customers) — same idea as your SCD1 union, just moved earlier since both buckets need fresh keys

In [0]:
df_inserts = df_new.unionByName(df_changed_new_version)
df_inserts.limit(5).display()

## Step 13: Generate surrogate keys (identical pattern to your SCD1 notebook)

In [0]:
df_inserts = df_inserts.withColumn("DimCustomerKey", monotonically_increasing_id() + lit(1))

In [0]:
df_inserts.display()

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0
else:
    df_maxsur = spark.sql("select max(DimCustomerKey) as max_surrogate_key from databricks_cata.gold.DimCustomersSCD2")
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']
    if max_surrogate_key is None:
        max_surrogate_key = 0

In [0]:
df_inserts = df_inserts.withColumn("DimCustomerKey", lit(max_surrogate_key) + col("DimCustomerKey"))
df_inserts = df_inserts.withColumn("merge_key", lit(None).cast("string"))
df_inserts.limit(5).display()

## Step 14: NEW for SCD2 — build the combined merge source

This is the standard SCD2 trick: stack the "expire" rows (which carry a real `merge_key`
so they MATCH an existing row) on top of the "insert" rows (which carry a `null` merge_key
so they can NEVER match — forcing the MERGE to insert them as new rows).

`df_expire` only needs `DimCustomerKey` + `merge_key` since it's only used to flip `is_current`.


In [0]:
expire_cols = [c for c in df_inserts.columns if c not in ("DimCustomerKey", "merge_key")]
df_expire_padded = df_expire
for c, dtype in df_inserts.select(*expire_cols).dtypes:
    df_expire_padded = df_expire_padded.withColumn(c, lit(None).cast(dtype))

df_merge_source = df_inserts.select("DimCustomerKey", "merge_key", *expire_cols)\
    .unionByName(df_expire_padded.select("DimCustomerKey", "merge_key", *expire_cols))

df_merge_source.display()

## Step 15: MERGE into the gold SCD2 table (same table-exists check as SCD1, different merge conditions)

- `whenMatched` (merge_key matches an existing key, i.e. an expire row) → set `is_current = 0` and stamp `effective_end_date`
- `whenNotMatched` (merge_key is null, i.e. an insert row) → insert the full new row

On the very first run (table doesn't exist) we just write everything out as current rows, same as your SCD1 init-load `else` branch.

In [0]:
if spark.catalog.tableExists("databricks_cata.gold.DimCustomersSCD2"):
    dlt_obj = DeltaTable.forPath(spark, "abfss://gold@customersprojectete.dfs.core.windows.net/DimCustomersSCD2")

    dlt_obj.alias("tgt").merge(
        df_merge_source.alias("src"),
        "tgt.DimCustomerKey = src.merge_key"
    ).whenMatchedUpdate(set={
        "is_current": lit(0),
        "effective_end_date": current_timestamp(),
        "update_date": current_timestamp()
    }).whenNotMatchedInsertAll().execute()

else:
    df_inserts.drop("merge_key").write.mode("overwrite")\
        .format("delta")\
        .option("path", "abfss://gold@customersprojectete.dfs.core.windows.net/DimCustomersSCD2")\
        .saveAsTable("databricks_cata.gold.DimCustomersSCD2")

In [0]:
%sql
select * from databricks_cata.gold.dimcustomersscd2 order by customer_id, effective_start_date

In [0]:
%sql
UPDATE databricks_cata.silver.customers_silver
SET city = 'chikodi'
WHERE customer_id = 'C00001';

In [0]:
%sql
select * from databricks_cata.silver.customers_silver
order by customer_id asc
limit 5

In [0]:
%sql
select * from databricks_cata.silver.customers_silver
where customer_id = "C00001"